# CSC2042S Assignment 4: Neural Networks

**Student Number:** `FFLADA001` &nbsp;&nbsp; **Name:** `Adam Foflonker`

### Pipeline:
1. **Data Loading & Preprocessing**
2. **Building & Training a Baseline Model**
3. **Hyperparameter Optimization**
4. **Analysis & Evaluation**

---

## Setup

Install required packages if missing:

``` bash
pip install torch torchvision matplotlib scikit-learn pandas seaborn numpy
```

The notebook uses `torchvision.datasets.FashionMNIST(download = True)` by default for the dataset.

---


## Imports


In [ ]:
import time
import copy
import random
import warnings

import numpy as np 
import pandas as pd

import matplotlib.pyplot as plt 
import seaborn as sns

import torch 
from torch import nn
from torch import optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets
from torchvision import transforms
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


## Global Constants, State & Flags

In [ ]:
BATCH_SIZE: int = 128
BASELINE_EPOCHS: int = 10
BASELINE_LR: float = 0.001
RANDOM_SEED: int = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
 
if torch.cuda.is_available(): 
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Suppress ConvergenceWarning from scikit-learn
warnings.filterwarnings("ignore", category = UserWarning)


## 1) Data Loading & Preprocessing

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

def get_dataloaders(batch_size = BATCH_SIZE, val_split = 0.1, data_dir = "./data"):
    train_full = datasets.FashionMNIST(root = data_dir, train = True, download = True, transform = transform)
    test = datasets.FashionMNIST(root = data_dir, train = False, download = True, transform = transform)
    
    n_val = int(len(train_full) * val_split)
    n_train = len(train_full) - n_val
    
    train_ds, val_ds = random_split(train_full, [n_train, n_val])
    train_loader = DataLoader(train_ds, batch_size = batch_size, shuffle = True, num_workers = 2, pin_memory = True)
    val_loader = DataLoader(val_ds, batch_size = batch_size, shuffle = False, num_workers = 2, pin_memory = True)
    test_loader = DataLoader(test, batch_size = batch_size, shuffle = False, num_workers = 2, pin_memory = True)
    
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = get_dataloaders(batch_size = BATCH_SIZE)
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")


## 2) Building & Training a Baseline Model

### 2.1) Multi-Layer Perceptron class defininition

This `MLP` class has the following architecture:
- Input Layer (a flattening layer)
- Hidden Layer 1: 128 neurons, ReLU activation
- Hidden Layer 2: 64 neurons, ReLU activation
- Output Layer: 10 neurons (one for each class)

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_size = 28 * 28, hidden1 = 128, hidden2 = 64, num_classes = 10, activation = nn.ReLU, dropout = 0.0):
        super().__init__()
        
        layers = [nn.Flatten(), nn.Linear(input_size, hidden1), activation()]
        if dropout > 0: 
            layers.append(nn.Dropout(dropout))
            
        layers += [nn.Linear(hidden1, hidden2), activation()]
        if dropout > 0: 
            layers.append(nn.Dropout(dropout))
            
        layers.append(nn.Linear(hidden2, num_classes))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x): return self.net(x)


### 2.2) Model Training Implementation

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0 
    correct = 0 
    total = 0
    
    for xb,yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
        _, preds = out.max(1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)
        
    return running_loss / total, correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0 
    correct = 0 
    total = 0
    all_preds = [] 
    all_targets = []
    
    with torch.no_grad():
        for xb,yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            running_loss += loss.item() * xb.size(0)
            _, preds = out.max(1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
            all_preds.append(preds.cpu().numpy()) 
            all_targets.append(yb.cpu().numpy())
            
    if len(all_preds) > 0:
        all_preds = np.concatenate(all_preds) 
        all_targets = np.concatenate(all_targets)
    else:
        all_preds = np.array([]) 
        all_targets = np.array([])
        
    return running_loss / total, correct / total, all_preds, all_targets

def train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs = BASELINE_EPOCHS, scheduler = None):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_acc = 0.0 
    best_model_wts = copy.deepcopy(model.state_dict())
    
    for e in range(epochs):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
        
        if scheduler is not None:
            try: 
                scheduler.step(val_loss)
            except: 
                scheduler.step()
                
        history["train_loss"].append(tr_loss) 
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(val_loss) 
        history["val_acc"].append(val_acc)
        
        if val_acc > best_acc:
            best_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            
        print(f"Epoch {e + 1}/{epochs}: train_loss = {tr_loss:.4f} | train_acc = {tr_acc:.4f} | val_loss = {val_loss:.4f} | " +
              f"val_acc = {val_acc:.4f} | time = {(time.time() - t0):.1f}s")
        
    model.load_state_dict(best_model_wts)
    return model, history


### 2.3) Baseline Model Training & Evaluation

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(batch_size = BATCH_SIZE, val_split = 0.1)
baseline_model = MLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(baseline_model.parameters(), lr = BASELINE_LR)

print(f"Training baseline model for {BASELINE_EPOCHS} epochs, this may take a while...")

baseline_model, baseline_history = train_model(baseline_model, train_loader, val_loader, criterion, optimizer, device, epochs = BASELINE_EPOCHS)
val_loss, val_acc, val_preds, val_targets = evaluate(baseline_model, val_loader, criterion, device)

print(f"Baseline final validation accuracy: {val_acc:.4f}")
baseline_epoch_df = pd.DataFrame({
    "epoch":        list(range(1, len(baseline_history["train_acc"]) + 1)),
    "train_acc":    baseline_history["train_acc"],
    "val_acc":      baseline_history["val_acc"],
    "train_loss":   baseline_history["train_loss"],
    "val_loss":     baseline_history["val_loss"] 
    })
display(baseline_epoch_df.style.format({"train_acc": "{:.4f}", "val_acc": "{:.4f}", "train_loss": "{:.4f}", "val_los": "{:.4f}"}))


## 3) Hyperparameter Optimization

The **learning rate**, **optimizer** and **dropout** is tuned separately for each language. This is done using a *grid search* to find the best combination of the 3 parameters based on validation accuracy.

In [ ]:
learning_rates = [0.1, 0.01, 0.001]
optimizers = ["SGD", "Adam", "RMSprop"]
dropouts = [0.0, 0.25, 0.5]

def make_optimizer(name, params, lr):
    if name == "SGD": 
        return optim.SGD(params, lr = lr, momentum = 0.9)
    if name == "Adam": 
        return optim.Adam(params, lr = lr)
    if name == "RMSprop": 
        return optim.RMSprop(params, lr = lr)
    raise ValueError(name)

results = []
best_overall = {"val_acc": 0.0, "lr": 0.0,"optimizer": "", "dropout": 0.0, "history": {}, "model": None}
max_epochs = 6  # Reduced for speed

for lr in learning_rates:
    for opt_name in optimizers:
        for dp in dropouts:
            cfg = {"lr": lr, "optimizer": opt_name, "dropout": dp}
            print(f"Running {cfg}")

            model = MLP(dropout = dp).to(device)
            criterion = nn.CrossEntropyLoss()
            optimizer = make_optimizer(opt_name, model.parameters(), lr)
            model, history = train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs=max_epochs)
            val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
            results.append({"lr": lr, "optimizer": opt_name, "dropout": dp, "val_acc": val_acc, "val_loss": val_loss, "history": history, "model": model})
            
            if val_acc > best_overall["val_acc"]:
                best_overall = {"val_acc": val_acc, "lr": lr, "optimizer": opt_name, "dropout": dp, "history": history, "model": copy.deepcopy(model)}
            
            print(f"Config result val_acc = {val_acc:.4f}\n")


results_df = pd.DataFrame([{"lr": r["lr"], "optimizer": r["optimizer"], "dropout": r["dropout"], "val_acc": r["val_acc"], "val_loss": r["val_loss"]} for r in results])
results_df = results_df.sort_values("val_acc", ascending = False).reset_index(drop = True)

# Expose best model and history
best_trained_model = best_overall["model"]
best_history = best_overall["history"]

print("Best configuration (inline):")
print({"lr": best_overall["lr"], "optimizer": best_overall["optimizer"], "dropout": best_overall["dropout"], "val_acc": best_overall["val_acc"]})
display(results_df.style.format({"val_acc": "{:.4f}", "val_loss": "{:.4f}"}))


## 4) Analysis & Evaluation

In [ ]:
# Ranked table & best config summary
print("=== Hyperparameter Results (sorted by val_acc) ===")
display(results_df.style.format({"val_acc": "{:.4f}", "val_loss": "{:.4f}"}))

best_cfg = {
    "lr":           best_overall["lr"],
    "optimizer":    best_overall["optimizer"],
    "dropout":      best_overall["dropout"],
    "val_acc":      best_overall["val_acc"]
}

print("\n=== Best Configuration Found ===")
print(f"Learning rate: {best_cfg["lr"]}")
print(f"Optimizer:     {best_cfg["optimizer"]}")
print(f"Dropout:       {best_cfg["dropout"]}")
print(f"Val accuracy:  {best_cfg["val_acc"]:.4f}")

# Heatmaps: impact of learning rate vs dropout for each optimizer
if "results_df" in globals():
    for opt in results_df["optimizer"].unique():
        sub = results_df[results_df["optimizer"] == opt]
        pivot = sub.pivot(index = "dropout", columns = "lr", values = "val_acc")
        plt.figure(figsize = (6, 4))
        sns.heatmap(pivot, annot = True, fmt = ".4f", cbar_kws = {"label": "val_acc"})
        plt.title(f"Validation Accuracy heatmap (optimizer = {opt})")
        plt.ylabel("dropout")
        plt.xlabel("learning rate")
        plt.suptitle("")
        plt.show()
        print(f"[Figure] Heatmap for optimizer = {opt}: rows = dropout, columns = learning rate")

# Loss curves: baseline vs best
baseline_hist = baseline_history
bh = best_history
plt.figure(figsize = (8, 5))
plt.plot(baseline_hist["val_loss"], label = "Baseline val_loss", linestyle = "--")
plt.plot(baseline_hist["train_loss"], label = "Baseline train_loss", linestyle = ":")
plt.plot(bh["val_loss"], label = "Best val_loss")
plt.plot(bh["train_loss"], label = "Best train_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss curves: Baseline vs Best")
plt.legend()
plt.show()
print("[Figure] Loss curves: Baseline vs Best model")

# Confusion matrix for the best model on the validation set
bm = best_trained_model
_, _, preds, targets = evaluate(bm, val_loader, criterion, device)
cm = confusion_matrix(targets, preds)
disp = ConfusionMatrixDisplay(cm, display_labels = [str(i) for i in range(10)])
fig, ax = plt.subplots(figsize = (8, 8))
disp.plot(ax = ax, cmap = "viridis", xticks_rotation = "vertical")
plt.title("Confusion Matrix - Best Model on Validation Set")
plt.show()
print("[Figure] Confusion matrix for the best model on the validation set")

# Epoch accuracy tables
baseline_df = pd.DataFrame({
    "epoch":        list(range(1, len(baseline_history["train_acc"]) + 1)),
    "train_acc":    baseline_history["train_acc"],
    "val_acc":      baseline_history["val_acc"]
    })
best_df = pd.DataFrame({
    "epoch":        list(range(1, len(best_history["train_acc"]) + 1)),
    "train_acc":    best_history["train_acc"],
    "val_acc":      best_history["val_acc"]
    })

print("\n=== Baseline Epoch Accuracies ===")
display(baseline_df.style.format({"train_acc": "{:.4f}", "val_acc": "{:.4f}"}))

print("\n=== Best Model Epoch Accuracies ===")
display(best_df.style.format({"train_acc": "{:.4f}", "val_acc": "{:.4f}"}))
